<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Ablation_Study_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [6]:
!git clone https://github.com/suman-poola/Random-Forrest-s-p500.git

%cd Random-Forrest-s-p500

from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

Cloning into 'Random-Forrest-s-p500'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 102 (delta 59), reused 2 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 18.43 MiB | 11.64 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Random-Forrest-s-p500/Random-Forrest-s-p500


In [8]:
from Data_Frame import build_dataset

df, x_test, x_train, y_train, y_test, encoding_table = build_dataset()

Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:00<00:00, 171MB/s] 


/content/Random-Forrest-s-p500/Random-Forrest-s-p500/Data_Frame.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period=time_period)
[*********************100%***********************]  503 of 503 completed
/content/Random-Forrest-s-p500/Random-Forrest-s-p500/Data_Frame.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


In [16]:
import torch


device = torch.device("cpu")

features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume','Price change per volume','id','mk cap']
x = df[features]
target = ['future return']

In [17]:
def make_sequences(window, ticker_col='ticker', date_col='date'):
    X_list, y_list = [], []

    for ticker, group in df.groupby(target):
        group = group.sort_values(date_col)
        feat_arr = group[features].values
        target_arr = group[ticker_col].values

        n_days = len(group)
        for i in range(n_days - window):
            X_list.append(feat_arr[i : i + window])
            y_list.append(target_arr[i + window])
    return np.array(X_list), np.array(y_list)

In [18]:
X_seq, y_seq = make_sequences(10)
X_seq.shape
y_seq.shape

KeyboardInterrupt: 

In [26]:
train_pct = 0.8
X_train, y_train = [], []
X_test, y_test = [], []
for ticker, group in df.groupby('ticker'):
    group = group.sort_values('date')
    feat_arr = group[features].values
    target_arr = group[target].values

    n_days = len(group)


    ticker_X, ticker_y = [], []
    for i in range(n_days - 10 ): #10 is the window, be sure to change it if you change the window
         ticker_X.append(feat_arr[i : i + 10])
         ticker_y.append(target_arr[i + 10])

    n_seqs = len(ticker_X)
    split_idx = int(train_pct * n_seqs)

    X_train.extend(ticker_X[:split_idx])
    y_train.extend(ticker_y[:split_idx])
    X_test.extend(ticker_X[split_idx:])
    y_test.extend(ticker_y[split_idx:])

x_train = np.array(X_train)
y_train = np.array(y_train)
x_test = np.array(X_test)
y_test = np.array(y_test)

print("train:", x_train.shape, y_train.shape)
print("test: ", x_test.shape, y_test.shape)

train: (275541, 10, 18) (275541, 1)
test:  (69382, 10, 18) (69382, 1)


In [25]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(279565, 18)
(275541,)
(70388, 18)
(69382,)


In [27]:
import numpy as np
for i, f in enumerate(features):
    n_inf = np.isinf(x_train[:, :, i]).sum()
    if n_inf > 0:
        print(f, n_inf)

In [28]:
id_idx = features.index('id')
cont_idx = [i for i, f in enumerate(features) if f != 'id']
n_cont = len(cont_idx)


scaler = StandardScaler()
scaler.fit(x_train[:, :, cont_idx].reshape(-1, n_cont))

x_train_scaled = scaler.transform(x_train[:, :, cont_idx].reshape(-1, n_cont)).reshape(x_train.shape[0], x_train.shape[1], n_cont)
x_test_scaled = scaler.transform(x_test[:, :, cont_idx].reshape(-1, n_cont)).reshape(x_test.shape[0], x_test.shape[1], n_cont)

id_train = x_train[:, 0, id_idx].astype(np.int64)
id_test = x_test[:, 0, id_idx].astype(np.int64)
n_tickers = int(max(id_train.max(), id_test.max())) + 1

print(x_train_scaled.shape, id_train.shape, n_tickers)

(275541, 10, 17) (275541,) 503


In [29]:
class StockSequenceDataset(Dataset):
    def __init__(self, X, ids, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.ids[idx], self.y[idx]

train_ds = StockSequenceDataset(x_train_scaled, id_train, y_train)
test_ds = StockSequenceDataset(x_test_scaled, id_test, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, ib, yb = next(iter(train_loader))

In [30]:
class StockLSTM(nn.Module):
    def __init__(self, n_cont_features, n_tickers, emb_dim=16, hidden_size=64,
                 num_layers=1, dropout=0.3, emb_dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_tickers, emb_dim)
        self.emb_drop = nn.Dropout(emb_dropout)
        self.rnn = nn.LSTM(
            input_size=n_cont_features + emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,  # inter-layer dropout only applies when num_layers > 1
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, ids):
        e = self.emb_drop(self.emb(ids))                      # (B, emb_dim)
        e = e.unsqueeze(1).expand(-1, x.size(1), -1)          # (B, T, emb_dim)
        out, _ = self.rnn(torch.cat([x, e], dim=-1))          # (B, T, hidden)
        last_step = self.drop(out[:, -1, :])
        return self.fc(last_step)

model = StockLSTM(n_cont_features=n_cont, n_tickers=n_tickers,
                  emb_dim=16, hidden_size=64, num_layers=1,
                  dropout=0.3, emb_dropout=0.1).to(device)

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

window = 10
train_pct = 0.8
n_epochs = 5

results = []

for removed_feature in features:
    current_features = [f for f in features if f != removed_feature]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for ticker, group in df.groupby('ticker'):
        group = group.sort_values('date')

        feat_arr = group[current_features].values
        target_arr = group['future return'].values

        n_days = len(group)

        ticker_X = []
        ticker_y = []

        for i in range(n_days - window):
            ticker_X.append(feat_arr[i:i + window])
            ticker_y.append(target_arr[i + window])

        split_idx = int(train_pct * len(ticker_X))

        X_train.extend(ticker_X[:split_idx])
        y_train.extend(ticker_y[:split_idx])
        X_test.extend(ticker_X[split_idx:])
        y_test.extend(ticker_y[split_idx:])

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    id_idx = current_features.index('id')
    cont_idx = [i for i, f in enumerate(current_features) if f != 'id']

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train[:, :, cont_idx].reshape(-1, len(cont_idx))
    ).reshape(X_train.shape[0], X_train.shape[1], len(cont_idx))

    X_test_scaled = scaler.transform(
        X_test[:, :, cont_idx].reshape(-1, len(cont_idx))
    ).reshape(X_test.shape[0], X_test.shape[1], len(cont_idx))

    id_train = X_train[:, 0, id_idx].astype(np.int64)
    id_test = X_test[:, 0, id_idx].astype(np.int64)
    n_tickers = int(max(id_train.max(), id_test.max())) + 1

    train_ds = StockSequenceDataset(X_train_scaled, id_train, y_train)
    test_ds = StockSequenceDataset(X_test_scaled, id_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    model = StockLSTM(
        n_cont_features=len(cont_idx),
        n_tickers=n_tickers,
        emb_dim=16,
        hidden_size=64,
        num_layers=1,
        dropout=0.3,
        emb_dropout=0.1
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(n_epochs):
        model.train()

        for xb, ib, yb in train_loader:
            xb, ib, yb = xb.to(device), ib.to(device), yb.to(device)

            optimizer.zero_grad()

            pred = model(xb, ib)
            loss = criterion(pred, yb)

            loss.backward()
            optimizer.step()

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, ib, yb in test_loader:
            xb, ib, yb = xb.to(device), ib.to(device), yb.to(device)

            pred = model(xb, ib)
            loss = criterion(pred, yb)

            total_loss += loss.item() * xb.size(0)

    test_mse = total_loss / len(test_ds)

    results.append({
        'removed_feature': removed_feature,
        'test_mse': test_mse
    })

    print(f"removed {removed_feature:25s} error {test_mse:.6f}")

results_df = pd.DataFrame(results)

Removed: close                     | Test MSE: 6.548235
Removed: return                    | Test MSE: 6.499433
Removed: Lag 1                     | Test MSE: 6.557782
Removed: Lag 2                     | Test MSE: 6.533990
Removed: Lag 3                     | Test MSE: 6.579569
Removed: Lag 5                     | Test MSE: 6.555770
Removed: volatility 5              | Test MSE: 6.525590
Removed: volatility 10             | Test MSE: 6.558047
Removed: volatility 20             | Test MSE: 6.596700
Removed: volatility 30             | Test MSE: 6.527213
Removed: SMA 10                    | Test MSE: 6.552141
Removed: SMA 20                    | Test MSE: 6.547220
Removed: SMA 30                    | Test MSE: 6.586323
Removed: SMA 50                    | Test MSE: 6.558045
Removed: volume                    | Test MSE: 6.578684
Removed: Price change per volume   | Test MSE: 6.537074


ValueError: 'id' is not in list